# L5 · Flagship — Align a Tiny Assistant (RLHF with PPO)

**RL for LLMs / RLHF — Vizuara AI Labs · Dr. Rajat Dandekar**

This is the whole pipeline, end to end. You have a reward model (Lab A). Now you run **Stage 3: PPO** — the same algorithm from Lecture 4 — to push a GPT-2 policy toward higher reward, on a leash to a frozen reference. Then you **break the leash (β=0)** and watch it reward-hack.

The policy is a `AutoModelForCausalLMWithValueHead` — literally the lecture's **two heads on one trunk** (actor + value). TRL runs the PPO machinery; **you implement the reward signal** that drives it.

What you should see:
- **With the leash (β=0.2):** reward rises modestly, KL stays bounded (~single digits), answers stay coherent.
- **Without the leash (β=0):** reward rises *more*, but KL explodes (~40+) and the text degenerates into reward-hacked gibberish.

> Runtime: **GPU**. ~15–20 minutes (runs PPO twice).

In [ ]:
!pip -q install "transformers==4.45.2" "trl==0.11.4" "datasets==3.0.1" "accelerate>=0.34,<1.1" >/dev/null

In [ ]:
# ✅ PROVIDED — the reward model from Lab A (load ./rm, else quick-train on HH)
import os, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset
device = "cuda" if torch.cuda.is_available() else "cpu"
rm_tok = AutoTokenizer.from_pretrained("gpt2"); rm_tok.pad_token = rm_tok.eos_token; rm_tok.truncation_side = "left"
if os.path.isdir("rm"):
    rm = AutoModelForSequenceClassification.from_pretrained("rm").to(device).eval()
    print("loaded your Lab A reward model")
else:
    print("quick-training a reward model on HH (~3 min)...")
    rm = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=1)
    rm.config.pad_token_id = rm_tok.eos_token_id; rm.to(device)
    _d = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=0).select(range(4000))
    def _sc(t):
        e = rm_tok(t, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        return rm(**e).logits.squeeze(-1)
    _o = torch.optim.AdamW(rm.parameters(), lr=2e-5)
    for i in range(0, 4000, 8):
        rc, rr = _sc(_d["chosen"][i:i+8]), _sc(_d["rejected"][i:i+8])
        L = -F.logsigmoid(rc-rr).mean(); _o.zero_grad(); L.backward(); _o.step()
    rm.eval(); print("done.")

## 1 · The policy (actor + value head) and the prompts

In [ ]:
# ✅ PROVIDED — policy with a value head, and a dataset of HH prompts
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler
tok = AutoTokenizer.from_pretrained("gpt2"); tok.pad_token = tok.eos_token; tok.truncation_side = "left"

def hh_prompt(t):
    cut = t.rfind("Assistant:"); return t[:cut+len("Assistant:")] if cut != -1 else t
raw = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=0).select(range(2000))
ds = Dataset.from_dict({"query": [hh_prompt(t) for t in raw["chosen"]]})
ds = ds.map(lambda s: {"input_ids": tok(s["query"], truncation=True, max_length=96).input_ids, "query": s["query"]})
ds = ds.filter(lambda r: 4 <= len(r["input_ids"]) <= 96); ds.set_format("torch")
print("prompts:", len(ds), "| example:", ds[0]["query"][-80:])

## 2 · 📝 TODO — the reward signal

The reward for one generated answer is the **reward model's score** of the prompt-plus-answer. Implement it: given a list of full texts, return a list of scalar reward **tensors** (one per text).

In [ ]:
@torch.no_grad()
def compute_rewards(texts):
    # 📝 TODO: score each "prompt + response" string with the reward model `rm`
    #   (tokenize with rm_tok -> rm(...).logits -> one scalar per text) and return
    #   a LIST of 1-D CPU tensors, e.g. [tensor(0.7), tensor(-0.2), ...].
    raise NotImplementedError("return per-sample reward tensors from the reward model")

## 3 · The PPO loop

`run_rlhf(init_kl_coef, steps)` builds a PPO trainer with a given KL-leash strength and runs the loop: generate → score with **your** `compute_rewards` → `ppo_trainer.step`. `init_kl_coef=0.2` is the normal leash; `0.0` removes it.

In [ ]:
# ✅ PROVIDED — one RLHF run (uses your compute_rewards)
def distinct2(texts):
    bg, tot = set(), 0
    for t in texts:
        w = t.split()
        for i in range(len(w)-1): bg.add((w[i], w[i+1])); tot += 1
    return len(bg)/max(tot, 1)

def run_rlhf(init_kl_coef=0.2, steps=60, seed=0):
    torch.manual_seed(seed)
    policy = AutoModelForCausalLMWithValueHead.from_pretrained("gpt2")   # actor + value head
    cfg = PPOConfig(model_name="gpt2", learning_rate=1.41e-5, batch_size=32, mini_batch_size=8,
                    ppo_epochs=4, init_kl_coef=init_kl_coef, adap_kl_ctrl=(init_kl_coef > 0), seed=seed)
    ppo = PPOTrainer(cfg, policy, None, tok, dataset=ds,
                     data_collator=lambda d: dict((k, [x[k] for x in d]) for k in d[0]))
    gen_kw = dict(min_length=-1, top_k=0.0, top_p=1.0, do_sample=True, pad_token_id=tok.eos_token_id)
    lens = LengthSampler(16, 24)
    rew, kl, d2, it = [], [], [], iter(ppo.dataloader)
    samples = {}
    for step in range(steps):
        try: batch = next(it)
        except StopIteration: it = iter(ppo.dataloader); batch = next(it)
        q = batch["input_ids"]
        resp = ppo.generate(q, return_prompt=False, length_sampler=lens, **gen_kw)
        rtxt = [tok.decode(r) for r in resp]
        rewards = compute_rewards([qq + rr for qq, rr in zip(batch["query"], rtxt)])   # <- your TODO
        stats = ppo.step(q, resp, rewards)
        rew.append(float(torch.stack(rewards).mean())); kl.append(float(stats["objective/kl"])); d2.append(distinct2(rtxt))
        if step == 0: samples["first"] = rtxt[:3]
        if step == steps-1: samples["last"] = rtxt[:3]
        if step % 10 == 0: print(f"  step {step:3d}  reward {rew[-1]:+.3f}  kl {kl[-1]:.1f}  distinct2 {d2[-1]:.3f}")
    return {"reward": rew, "kl": kl, "distinct2": d2, "samples": samples}

## 4 · Run it — with the leash (normal RLHF)

In [ ]:
# ✅ PROVIDED
print("=== NORMAL RLHF (beta = 0.2) ===")
normal = run_rlhf(init_kl_coef=0.2, steps=60)
print("\nfinal answers:", normal["samples"]["last"])

## 5 · Break the leash — the reward-hacking ablation

Set the KL coefficient to **0** and run again. The policy is now free to chase the reward model into its blind spots (the ones you found in Lab C).

In [ ]:
# ✅ PROVIDED
print("=== ABLATION (beta = 0, no KL leash) ===")
hacked = run_rlhf(init_kl_coef=0.0, steps=60)
print("\nfinal answers:", hacked["samples"]["last"])

In [ ]:
# ✅ PROVIDED — compare reward vs KL for both runs
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(normal["reward"], label="β=0.2 (leash)", c="#3D5A4A")
ax[0].plot(hacked["reward"], label="β=0 (hacking)", c="#8B3A3A")
ax[0].set_title("reward-model score"); ax[0].set_xlabel("PPO step"); ax[0].legend()
ax[1].plot(normal["kl"], label="β=0.2 (leash)", c="#3D5A4A")
ax[1].plot(hacked["kl"], label="β=0 (hacking)", c="#8B3A3A")
ax[1].set_title("KL to reference"); ax[1].set_xlabel("PPO step"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6 · Questions to answer

1. The ablation reaches **higher** reward than the leashed run — yet its answers are worse. Reconcile this with what the reward model is actually measuring. (Tie it to Lab C.)
2. Look at the two KL curves. What does the leash physically do to the policy's movement, and why does that protect quality?
3. The policy here is an `AutoModelForCausalLMWithValueHead`. Which head is being optimized by the clipped objective, and which by the value loss? (Connect to Lab D.)
4. RLHF holds several models at once. List them and mark which are **frozen** vs **trainable** in this notebook.
5. **Stretch:** raise `steps` and watch the β=0 run degrade further. At what KL does the text stop being English?
